# Synthesizability Scoring

Post-hoc AiZynthFinder retrosynthesis scoring on already-generated variant CSVs.
Writes an `is_synth_depth{MAX_DEPTH}` column back to each `*_scores.csv` without re-running generation.

**Flow:**
1. Load all method CSVs from `GEN_DIR`
2. Property gate → collect unique candidates (tanimoto ≥ τ_t, desirability ≥ τ_d)
3. Run AiZynthFinder with checkpoint resuming → saves `synth_depth{N}.json`
4. Write `is_synth_depth{N}` back to each CSV (one column per depth; existing depth columns are preserved)

**Re-run for a different depth:** change `MAX_DEPTH` in the config cell and re-run Stages 2–3.

In [ ]:
import sys, json
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed
from concurrent.futures.process import BrokenProcessPool

import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.synth_parallel import worker_init, score_one

# ── Config ────────────────────────────────────────────────────────────────────
GEN_DIR = ROOT / 'data' / 'generation_stratified'

AIZYNTHFINDER_CONFIG = ROOT / 'data' / 'aizynthfinder' / 'config.yml'

# MAX_DEPTH: retrosynthetic search depth. Use 1 / 3 / 6 for Table 3.
MAX_DEPTH  = 6

# TIME_LIMIT: per-molecule wall-clock limit (seconds).
TIME_LIMIT = 120

# Checkpoint — one file per depth.
SYNTH_CKPT = GEN_DIR / f'synth_depth{MAX_DEPTH}.json'

METHODS = {
    'baseline':  GEN_DIR / 'baseline'  / 'baseline_scores.csv',
    'crem':      GEN_DIR / 'crem'      / 'crem_scores.csv',
    'libinvent': GEN_DIR / 'libinvent' / 'libinvent_scores.csv',
    'mmpdb':     GEN_DIR / 'mmpdb'     / 'mmpdb_scores.csv',
    'jtvae':     GEN_DIR / 'jtvae'     / 'jtvae_scores.csv',
}

TAU_T     = 0
TAU_D     = 0.5
N_WORKERS = 10
TIMEOUT   = TIME_LIMIT + 30   # future-level timeout

print(f'GEN_DIR    : {GEN_DIR}')
print(f'AiZynth cfg: {"OK" if AIZYNTHFINDER_CONFIG.exists() else "MISSING"}')
print(f'Max depth  : {MAX_DEPTH}')
print(f'Time limit : {TIME_LIMIT}s/mol')
print(f'Checkpoint : {SYNTH_CKPT.name} -> {"exists (will resume)" if SYNTH_CKPT.exists() else "fresh run"}')
print(f'Gate       : tanimoto >= {TAU_T}, desirability >= {TAU_D}')

GEN_DIR    : D:\AI4DD Project\CSC2541-Project\data\generation_stratified
AiZynth cfg: OK
Max depth  : 1
Time limit : 60s/mol
Checkpoint : synth_depth1.json -> fresh run
Gate       : tanimoto >= 0, desirability >= 0.5


---
## Stage 1 — Load CSVs & collect candidates

In [13]:
dfs = {}
for name, path in METHODS.items():
    if path.exists():
        df = pd.read_csv(path)
        df['method'] = name
        dfs[name] = df
        print(f'{name:<12}: {len(df):>6,} rows')
    else:
        print(f'{name:<12}: MISSING -> {path}')

# Collect unique candidates that pass the property gate
candidates = set()
for name, df in dfs.items():
    mask = (df['tanimoto'] >= TAU_T) & (df['desirability'] >= TAU_D)
    c = df.loc[mask, 'variant_smiles'].dropna().unique()
    candidates.update(c)
    print(f'  {name:<10}: {mask.sum():>5} pass gate -> {len(c):>5} unique')

candidates = list(candidates)
print(f'\nTotal unique candidates for AiZynthFinder: {len(candidates)}')

baseline    :  4,342 rows
crem        : 11,711 rows
libinvent   : 31,440 rows
mmpdb       : 10,641 rows
jtvae       : 10,536 rows
  baseline  :   299 pass gate ->   299 unique
  crem      :   968 pass gate ->   968 unique
  libinvent :   318 pass gate ->   272 unique
  mmpdb     :   714 pass gate ->   714 unique
  jtvae     :  1087 pass gate ->  1087 unique

Total unique candidates for AiZynthFinder: 3232


---
## Stage 2 — Run AiZynthFinder (with checkpoint resuming)

In [14]:
assert AIZYNTHFINDER_CONFIG.exists(), f'AiZynthFinder config not found: {AIZYNTHFINDER_CONFIG}'

synth: dict[str, bool] = {}
if SYNTH_CKPT.exists():
    with open(SYNTH_CKPT) as f:
        synth = json.load(f)
    print(f'Resumed checkpoint: {len(synth)} already scored')

pending = [s for s in candidates if s not in synth]
print(f'Remaining: {len(pending)} molecules to score')

if pending:
    # max_tasks_per_child=None: workers never restart mid-run, eliminating
    # the multi-minute pauses caused by reloading AiZynthFinder models.
    # Hung molecules are handled by the thread timeout in score_one.
    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        initializer=worker_init,
        initargs=(str(AIZYNTHFINDER_CONFIG), MAX_DEPTH, TIME_LIMIT),
        max_tasks_per_child=None,
    ) as pool:
        futures = {pool.submit(score_one, s): s for s in pending}
        with tqdm(total=len(pending), desc=f'depth={MAX_DEPTH}', unit='mol') as pbar:
            for fut in as_completed(futures):
                smi = futures[fut]
                try:
                    _, solved = fut.result(timeout=TIMEOUT)
                except FutureTimeoutError:
                    solved = False
                except BrokenProcessPool:
                    # A worker actually crashed (rare); save and restart
                    with open(SYNTH_CKPT, 'w') as f:
                        json.dump(synth, f)
                    raise
                except Exception:
                    solved = False
                synth[smi] = solved
                pbar.update(1)

    with open(SYNTH_CKPT, 'w') as f:
        json.dump(synth, f)

n_solved = sum(synth.values())
print(f'\nDone: {n_solved}/{len(synth)} synthesizable ({100*n_solved/max(len(synth),1):.1f}%)')

Remaining: 3232 molecules to score


depth=1:   0%|          | 0/3232 [00:00<?, ?mol/s]


Done: 762/3232 synthesizable (23.6%)


---
## Stage 3 — Write `is_synth_depth{N}` back to CSVs

Each depth run adds one column; columns from other depths are preserved.

In [15]:
col = f'is_synth_depth{MAX_DEPTH}'
for name, df in dfs.items():
    # Molecules not in synth dict did not pass the property gate -> False
    df[col] = df['variant_smiles'].map(synth).fillna(False)
    path = METHODS[name]
    df.to_csv(path, index=False)
    n = df[col].sum()
    print(f'{name:<12}: {n:>5}/{len(df)} synthesizable at depth {MAX_DEPTH} ({100*n/len(df):.1f}%)')

baseline    :    82/4342 synthesizable at depth 1 (1.9%)
crem        :   287/11711 synthesizable at depth 1 (2.5%)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2289289802.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth).fillna(False)
C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2289289802.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth).fillna(False)
C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2289289802.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(c

libinvent   :   158/31440 synthesizable at depth 1 (0.5%)
mmpdb       :   156/10641 synthesizable at depth 1 (1.5%)
jtvae       :   201/10536 synthesizable at depth 1 (1.9%)


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2289289802.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth).fillna(False)
C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2289289802.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth).fillna(False)


---
## Stage 4 — Summary

---
## Utility — Backfill all available depth checkpoints into CSVs

Run this once to ensure every available `synth_depthN.json` is reflected as an `is_synth_depthN` column in all `*_scores.csv` files.

In [18]:
DEPTH_VARIANTS = [1, 3, 6]

# Reload fresh copies of the CSVs so we don't overwrite in-memory edits from above
backfill_dfs = {name: pd.read_csv(path) for name, path in METHODS.items() if path.exists()}

for depth in DEPTH_VARIANTS:
    ckpt_path = GEN_DIR / f'synth_depth{depth}.json'
    col = f'is_synth_depth{depth}'
    if not ckpt_path.exists():
        print(f'depth {depth}: checkpoint missing, skipping')
        continue
    with open(ckpt_path) as f:
        synth_d = json.load(f)
    n_synth = sum(synth_d.values())
    print(f'depth {depth}: {len(synth_d)} scored, {n_synth} synthesizable')
    for name, df in backfill_dfs.items():
        df[col] = df['variant_smiles'].map(synth_d).fillna(False)

# Write back once with all depth columns present
for name, df in backfill_dfs.items():
    path = METHODS[name]
    df.to_csv(path, index=False)
    depth_cols = [c for c in df.columns if c.startswith('is_synth_depth')]
    summary_parts = [f'd{c.split("depth")[1]}={int(df[c].sum())}' for c in depth_cols]
    print(f'{name:<12}: {" | ".join(summary_parts)}')

depth 1: 3232 scored, 762 synthesizable
depth 3: 3232 scored, 2008 synthesizable
depth 6: checkpoint missing, skipping
baseline    : d3=157 | d1=82
crem        : d3=713 | d1=287


C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2494563055.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth_d).fillna(False)
C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2494563055.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df['variant_smiles'].map(synth_d).fillna(False)
C:\Users\Acmaro\AppData\Local\Temp\ipykernel_22132\2494563055.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_ob

libinvent   : d3=343 | d1=158
mmpdb       : d3=435 | d1=156
jtvae       : d3=631 | d1=201


In [17]:
col = f'is_synth_depth{MAX_DEPTH}'
rows = []
for name, df in dfs.items():
    gate = (df['tanimoto'] >= TAU_T) & (df['desirability'] >= TAU_D)
    hits = gate & df[col]
    rows.append({
        'method':           name,
        'total_variants':   len(df),
        'pass_property':    gate.sum(),
        'pass_synth':       hits.sum(),
        f'synth_rate_d{MAX_DEPTH}(%)': round(100 * hits.sum() / gate.sum(), 1) if gate.sum() > 0 else 0,
        'final_hits':       hits.sum(),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

   method  total_variants  pass_property  pass_synth  synth_rate_d1(%)  final_hits
 baseline            4342            299          82              27.4          82
     crem           11711            968         287              29.6         287
libinvent           31440            318         109              34.3         109
    mmpdb           10641            714         156              21.8         156
    jtvae           10536           1087         201              18.5         201
